# Module 1 demo: RAG for a literature review

**Topic:** AI for early lung cancer detection from low-dose CT.
**Corpus:** 272 PubMed abstracts. **Model:** Claude Haiku 5.5.

By the end of this notebook you will have a working system that answers
literature-review questions over the corpus and cites PMIDs. Every section
has a short explanation, then the code. Run it top to bottom.

RAG = **Retrieval-Augmented Generation**. The idea in one line: before asking
the LLM, *retrieve* the few documents that matter and paste them into the
prompt, so the model answers from your data instead of from memory.

Setup: copy `.env.example` to `.env` and put your `ANTHROPIC_API_KEY` in it.

In [1]:
from dotenv import load_dotenv
load_dotenv()

import anthropic
client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from .env
MODEL = "claude-haiku-5-5"

## 1. The problem: the model does not know your corpus

Ask Claude a question about the topic with no context. The answer will sound
reasonable and generic. It cannot tell you *which papers in your corpus* did
what, and it may cite things that do not exist. That is the gap RAG closes.

Watch for two tells in the answer: the model hedges about its own citations
("please verify these PMIDs"), and nothing ties the claims to *your* 272
abstracts. Also note the token counts: almost no input, lots of output. RAG
flips that ratio.

In [2]:
def llm(prompt, system=None):
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=system or "You are a concise research assistant.",
        messages=[{"role": "user", "content": prompt}],
        output_config={"effort": "low"},
    )
    text = "".join(b.text for b in response.content if b.type == "text")
    return text, response.usage

question = "Who has validated an AI system for lung cancer screening in a real screening cohort? Cite PMIDs."
answer, usage = llm(question)
print(answer)
print("\ntokens:", usage.input_tokens, "in /", usage.output_tokens, "out")

Several groups have validated AI systems for lung cancer risk or nodule assessment on screening data. The two I'm most confident about are:

1. **Sybil (Mikhael et al., MIT/Massachusetts General Hospital/Mass General Brigham)**: a deep learning model that predicts future lung cancer risk from a single low-dose CT. It was validated on NLST data and on an independent screening cohort from Mass General Brigham, which makes it one of the more clearly real-world validations. *J Clin Oncol.* 2023. PMID: 36972470.

2. **Ardila et al. (Google Health, with Northwestern and others)**: an end-to-end 3D deep learning model for malignancy risk from screening low-dose CT. It was trained on NLST and tested on held-out NLST data and an independent screening dataset, showing performance comparable to radiologists. *Nat Med.* 2019. PMID: 31192478.

Other validations exist, including studies in European screening programs such as NELSON and the Danish lung cancer screening trial, but I can't reliably rec

Notice two things. The answer is plausible but unverifiable, and any PMIDs it
gives are from memory, so they may be wrong. We want answers grounded in
abstracts we actually have.

## 2. Get the corpus from PubMed

This is the **ingestion** step: pull data from its source into a shape we can
search. PubMed exposes its database through the E-utilities API:

- `esearch` turns a query into a list of PMIDs
- `efetch` returns the full records for those PMIDs as XML

Real data is messy: some records have no abstract, some abstracts are split
into labeled sections, publication dates come in several formats. The parser
in `pubmed_ingest.py` handles that. Have a look at it, it is short.

For reproducibility everyone uses the **same frozen PMID list** in
`data/pmids.txt`. Below we first show the live search so you see where the
list came from, then fetch by the frozen list.

In [3]:
from pubmed_ingest import QUERY, search_pmids, read_pmids, load_corpus

print(QUERY)
live_pmids, total = search_pmids(QUERY)
print(f"PubMed reports {total} hits today")

pmids = read_pmids("data/pmids.txt")
print(f"Frozen list: {len(pmids)} PMIDs")

"lung cancer" AND ("deep learning" OR "artificial intelligence") AND ("low-dose CT" OR "low-dose computed tomography") AND screening


PubMed reports 278 hits today
Frozen list: 278 PMIDs


In [4]:
documents = load_corpus(pmids)   # efetch + parse + drop records without abstract
print(f"{len(documents)} documents with abstracts")

272 documents with abstracts


In [5]:
import json
print(json.dumps(documents[0], indent=1)[:1200])

{
 "pmid": "16156352",
 "title": "Computer-aided diagnostic scheme for distinction between benign and malignant nodules in thoracic low-dose CT by use of massive training artificial neural network.",
 "abstract": "Low-dose helical computed tomography (LDCT) is being applied as a modality for lung cancer screening. It may be difficult, however, for radiologists to distinguish malignant from benign nodules in LDCT. Our purpose in this study was to develop a computer-aided diagnostic (CAD) scheme for distinction between benign and malignant nodules in LDCT scans by use of a massive training artificial neural network (MTANN). The MTANN is a trainable, highly nonlinear filter based on an artificial neural network. To distinguish malignant nodules from six different types of benign nodules, we developed multiple MTANNs (multi-MTANN) consisting of six expert MTANNs that are arranged in parallel. Each of the MTANNs was trained by use of input CT images and teaching images containing the estima

In [6]:
# What does the corpus look like?
from collections import Counter
years = Counter(d["year"] for d in documents)
print("by year:", sorted(years.items()))
print("median abstract length:", sorted(len(d["abstract"]) for d in documents)[len(documents)//2], "chars")

by year: [(2005, 1), (2010, 1), (2014, 1), (2018, 3), (2019, 10), (2020, 12), (2021, 22), (2022, 33), (2023, 20), (2024, 28), (2025, 62), (2026, 79)]
median abstract length: 1867 chars


## 3. Search

Every search engine does the same thing: score each document for similarity
to the query, then return the top N. What differs is the similarity function.

Here we use **keyword (lexical) search** with `minsearch`, a tiny in-memory
engine. It matches on words, so "nodule" matches "nodule" but not "lesion".
Semantic (vector) search fixes that, and is a later module.

Three ideas to notice in the index definition:

- **text fields** are searched (`title`, `abstract`)
- **keyword fields** are for exact-match filtering (`pmid`, `journal`)
- **numeric fields** allow range filters (`year`)

We **boost** the title, since a word in the title is a stronger signal than
the same word buried in the abstract.

In [7]:
from pubmed_ingest import build_index
index = build_index(documents)

def show(results):
    for d in results:
        print(f"{d['pmid']}  {d['year']}  {d['first_author']:<16} {d['title'][:90]}")

query = "Which deep learning models predict nodule malignancy on low-dose CT?"
results = index.search(query, num_results=5, boost_dict={"title": 2.0, "abstract": 1.0})
show(results)

36634294  2023  Mikhael PG       Sybil: A Validated Deep Learning Model to Predict Future Lung Cancer Risk From a Single Lo
35137543  2022  Wu Z             Lung cancer risk prediction models based on pulmonary nodules: A systematic review.
40833260  2025  Obreja B         Characterizing the Impact of Training Data on Generalizability: Application in Deep Learni
41579047  2026  Chen SS          Evaluating the impact of deep learning-based image denoising on low-dose CT for lung cance
37870625  2024  Peters AA        Influence of CT dose reduction on AI-driven malignancy estimation of incidental pulmonary 


In [8]:
# Filtering: restrict to a year range. Same query, different answer set.
results_recent = index.search(
    query, num_results=5,
    boost_dict={"title": 2.0, "abstract": 1.0},
    filter_dict={"year": [(">=", 2022), ("<=", 2024)]},
)
show(results_recent)

36634294  2023  Mikhael PG       Sybil: A Validated Deep Learning Model to Predict Future Lung Cancer Risk From a Single Lo
35137543  2022  Wu Z             Lung cancer risk prediction models based on pulmonary nodules: A systematic review.
37870625  2024  Peters AA        Influence of CT dose reduction on AI-driven malignancy estimation of incidental pulmonary 
37526548  2023  Venkadesh KV     Prior CT Improves Deep Learning for Malignancy Risk Estimation of Screening-detected Pulmo
34969636  2022  Aquino GJ        Deep learning model to quantify left atrium volume on routine non-contrast chest CT and pr


## 4. Build the prompt

The LLM only sees what we put in the prompt. We split it in two:

- **Instructions** (the system prompt): fixed for every request. This is where
  we demand citations and forbid outside knowledge.
- **User prompt**: changes each time. It holds the question and the retrieved
  abstracts, formatted so the model can tell them apart.

Each context block carries the PMID, first author, year and journal, so the
model can attribute claims. Grounding the answer in this context is what
reduces hallucination.

In [9]:
from rag_helper import INSTRUCTIONS, PROMPT_TEMPLATE
print(INSTRUCTIONS)

You are a research assistant helping with a literature review on
AI for early lung cancer detection from CT.

Answer the question using only the abstracts provided in the context.
After each claim, cite the supporting abstract(s) by PMID in square
brackets, like [PMID 12345678]. Prefer naming the first author and year
when describing who did what.

If the context does not contain the information needed, say
"Not covered in the retrieved abstracts." Do not use outside knowledge.


In [10]:
def build_context(search_results):
    blocks = []
    for d in search_results:
        blocks.append(
            f"PMID {d['pmid']} | {d['first_author']} et al., {d['year']} | {d['journal']}\n"
            f"TITLE: {d['title']}\n"
            f"ABSTRACT: {d['abstract']}"
        )
    return "\n\n".join(blocks)

def build_prompt(question, search_results):
    return PROMPT_TEMPLATE.format(question=question, context=build_context(search_results))

prompt = build_prompt(query, results)
print(prompt[:1500], "\n...\n")
print(f"prompt length: {len(prompt)} characters")

QUESTION: Which deep learning models predict nodule malignancy on low-dose CT?

CONTEXT:
PMID 36634294 | Mikhael PG et al., 2023 | J Clin Oncol
TITLE: Sybil: A Validated Deep Learning Model to Predict Future Lung Cancer Risk From a Single Low-Dose Chest Computed Tomography.
ABSTRACT: PURPOSE: Low-dose computed tomography (LDCT) for lung cancer screening is effective, although most eligible people are not being screened. Tools that provide personalized future cancer risk assessment could focus approaches toward those most likely to benefit. We hypothesized that a deep learning model assessing the entire volumetric LDCT data could be built to predict individual risk without requiring additional demographic or clinical data.
METHODS: We developed a model called Sybil using LDCTs from the National Lung Screening Trial (NLST). Sybil requires only one LDCT and does not require clinical data or radiologist annotations; it can run in real time in the background on a radiology reading station. 

## 5. Call the LLM

Send the prompt. Then look at `usage`: the API reports how many tokens went in
and came out. Tokens are what you pay for, and the context we retrieved is
most of the input. Keep an eye on this number, it is the lever behind every
design choice in RAG (how many results, how long each is, whether to chunk).

Haiku 5.5 list price: $0.10 per million input tokens, $0.50 per million output.

In [11]:
answer, usage = llm(prompt, system=INSTRUCTIONS)
print(answer)
cost = (usage.input_tokens * 0.10 + usage.output_tokens * 0.50) / 1e6
print(f"\ntokens: {usage.input_tokens} in / {usage.output_tokens} out   cost: ${cost:.5f}")

Based on the retrieved abstracts, the answer is only partly covered. Two studies describe deep learning models that estimate nodule malignancy on low-dose CT, and the abstracts do not name specific architectures beyond "convolutional neural network."

- **Obreja B et al., 2025** trained a convolutional neural network on 16,077 annotated nodules from the NLST to estimate pulmonary nodule malignancy risk on LDCT. On external DLCST testing, it reached a mean AUC of 0.92 when trained on the full NLST dataset [PMID 40833260]. It also reached clinician-level performance on a size-matched cancer-enriched DLCST subset (AUC 0.82) with only 20% of the training data [PMID 40833260].

- **Peters AA et al., 2024** evaluated a commercially available lung cancer prediction convolutional neural network (LCP-CNN) that outputs a malignancy score from 1 to 10. Scores were stable at 25% dose but changed at 5% simulated dose, and the proportion of correctly classified nodules fell under a "rule-in" thresho

## 6. Wrap it up: `RAGBase`

Search, prompt, LLM. We will call that sequence many times, so it lives in a
class in `rag_helper.py`. The index and the client are constructor arguments,
so you can swap either without touching the rest. `rag()` returns the answer
plus token counts.

Now the interesting part. A literature review asks three kinds of question,
and plain top-5 retrieval is not equally good at all of them.

In [12]:
from rag_helper import RAGBase
rag = RAGBase(index, llm_client=client)

def ask(q, **kw):
    r = rag.rag(q, **kw)
    print(q, "\n")
    print(r.answer)
    print(f"\n[{r.input_tokens} in / {r.output_tokens} out, ${r.cost_usd:.5f}]")
    print("retrieved:", [d["pmid"] for d in r.search_results])
    print("=" * 80)
    return r

# (a) "Who has done what?"  -> fits RAG well: retrieve, then attribute.
r1 = ask("Who has validated an AI system for lung cancer screening in a real screening cohort?")

Who has validated an AI system for lung cancer screening in a real screening cohort? 

Cui X et al. (2022) is the only study in the retrieved abstracts that validated an AI system in an actual lung cancer screening program. They evaluated a deep learning-based computer-aided detection (DL-CAD) system in a Chinese LDCT screening program, using baseline screening scans from 180 individuals with nodules, mixed 1:1 with screenees without nodules (360 total) [PMID 34871936]. The DL-CAD system reached 90.1% sensitivity at 1.0 false positive per scan, compared with 76.0% sensitivity at 0.04 FP/scan for double reading [PMID 34871936]. The authors concluded it may assist radiologists in LDCT screening [PMID 34871936].

Other studies come close but do not clearly meet the criterion:

- **Guo J et al. (2020)** developed DeepLN and reported three retrospective clinical comparisons with human experts, but the abstract does not describe a screening cohort [PMID 33240975].
- **Jin Y et al. (2024)** (

In [13]:
# (b) "What is the most common dataset?"  -> does NOT fit top-5 retrieval.
# The model only sees 5 of 272 abstracts, so whatever it says about "most common"
# is a statement about those 5. Watch how confident it sounds anyway.
r2 = ask("What is the most common dataset used to train these models?")

What is the most common dataset used to train these models? 

The retrieved abstracts do not establish which dataset is most common overall, since none of them survey dataset usage. Among the abstracts provided, the **National Lung Screening Trial (NLST)** appears most often:

- Mukherjee et al., 2023 built their patient-level cancer prediction model on a subset of NLST (n = 1138) [PMID 36909593].
- Masquelin et al., 2025 selected NLST participants with solid nodules 4–20 mm for deep learning classification [PMID 38848168].
- Chen et al., 2025 created NLSTseg, a pixel-level annotated dataset built from NLST LDCT scans of 605 patients [PMID 40849438].

The **Lung Image Database Consortium (LIDC-IDRI)** is also a training source. Mukherjee et al., 2023 trained the first stage of their model, which detects nodules and assigns malignancy scores, on LIDC-IDRI [PMID 36909593].

The **Duke Lung Cancer Screening (DLCS) dataset** is described by Wang et al., 2025 as a reference dataset of annot

Aggregation questions ("most common", "how many studies", "trend over time")
need to look at the *whole* corpus. Retrieval picks a handful. Fixes belong to
later modules: extract structured fields from every abstract and count them,
or let an agent run many searches. For now the lesson is to recognise the
question type and not trust a top-5 answer to it.

In [14]:
# (c) "What is the overall implication?"  -> partially fits. More results help.
r3 = ask("What is the overall clinical implication of this literature for lung cancer screening programs?", num_results=10)

What is the overall clinical implication of this literature for lung cancer screening programs? 

Overall, the retrieved abstracts point toward a picture in which LDCT screening is established as effective, while AI and risk-based personalization are presented as promising tools that still need validation and implementation work. Most of the evidence is narrative or review-level, so the claims below are directional rather than definitive.

**Established benefit of LDCT screening.** The NLST and NELSON trials showed reduced mortality with low-dose CT compared with chest radiography or no screening [PMID 36563698, Adams SJ et al., 2023]. Lancaster HL et al. (2022) report a 20% reduction in lung cancer mortality versus chest X-ray in NLST and a 24% reduction versus no screening in NELSON, and note that the focus has therefore shifted to implementation research [PMID 35253286]. Stage-dependent survival is also cited: >75% 5-year survival at stage 1A versus <5% at stage 4 [PMID 35253286].



## 7. Check the citations

Grounding is only useful if you verify it. The model was told to cite PMIDs
from the context. We can check programmatically that every PMID it cited was
actually one it was shown. A citation that is not in the retrieved set is a
hallucination, even if the PMID happens to exist on PubMed.

In [15]:
from rag_helper import cited_pmids

def check_citations(result):
    shown = {d["pmid"] for d in result.search_results}
    cited = cited_pmids(result.answer)
    bad = [p for p in cited if p not in shown]
    print(f"cited {len(cited)} PMIDs, {len(cited) - len(bad)} verified, {len(bad)} not in context: {bad}")

for r in (r1, r2, r3):
    check_citations(r)

cited 5 PMIDs, 5 verified, 0 not in context: []
cited 5 PMIDs, 5 verified, 0 not in context: []
cited 10 PMIDs, 10 verified, 0 not in context: []


## 8. Persist the index

`minsearch` lives in memory. Stop the kernel and it is gone, so every restart
means fetching from PubMed and re-indexing. That is fine for 272 documents and
painful for a million.

The standard fix is to separate **ingestion** (write once to a persistent
index) from **querying** (read many times). `sqlitesearch` has the same API as
`minsearch` but stores everything in a SQLite file, using SQLite's built-in
full-text search. Nothing to install or run.

One thing to notice: the same query returns a different top 5 than minsearch
did in section 3. minsearch scores with TF-IDF and cosine similarity;
SQLite's full-text search uses BM25. Same documents, same pipeline, different
ranking function, different hits. Retrieval quality depends on the search
engine, which is what module 2 (vector search) and module 4 (evaluation) are
about.

In [16]:
import os
from sqlitesearch import TextSearchIndex

DB = "pubmed.db"
if os.path.exists(DB):
    os.remove(DB)

sqlite_index = TextSearchIndex(
    text_fields=["title", "abstract"],
    keyword_fields=["pmid", "journal"],
    numeric_fields=["year"],
    id_field="pmid",            # re-running updates instead of duplicating
    db_path=DB,
)
for d in documents:
    sqlite_index.add(d)
sqlite_index.close()
print("ingested", len(documents), "documents into", DB)

ingested 272 documents into pubmed.db


In [17]:
# A "new process": open the file and query it. No PubMed call, no re-indexing.
sqlite_index = TextSearchIndex(
    text_fields=["title", "abstract"],
    keyword_fields=["pmid", "journal"],
    numeric_fields=["year"],
    id_field="pmid",
    db_path=DB,
)
rag_persistent = RAGBase(sqlite_index, llm_client=client)
r = rag_persistent.rag("Which deep learning models predict nodule malignancy on low-dose CT?")
print(r.answer)
check_citations(r)
sqlite_index.close()

Three of the retrieved abstracts describe deep learning models that assess nodule malignancy on low-dose CT. A fourth, Sybil, is related but predicts future lung cancer risk from the whole scan rather than classifying a specific nodule.

**Models that classify nodule malignancy directly**

- **Obreja et al., 2025** trained a convolutional neural network (CNN) on 16,077 annotated nodules from the NLST to estimate pulmonary nodule malignancy risk on LDCT. External testing on the Danish Lung Cancer Screening Trial gave a mean AUC of 0.92 with the full NLST training set, and 0.82 on a size-matched cancer-enriched subset, where performance was noninferior to the average of 11 clinicians with 20% of the training data [PMID 40833260].

- **Wang H et al., 2025 (DECIPHER-NODL)** used a deep learning imaging model that processes LDCT scans for automated nodule detection and malignancy classification. It was combined with a cell-free DNA model through a stacked ensemble. The integrated model reac

## Wrap-up

You built: ingestion from a real API, a keyword index with boosting and
filtering, a grounded prompt, an LLM call with cost accounting, a reusable
`RAGBase`, a citation checker, and a persistent index.

You also saw the limit: top-k retrieval answers "who did what" well, answers
"what is the overall picture" partially, and cannot answer "what is most
common". Keep that in mind when you do the homework.

Next: `homework/homework.md`.